## Loads the raw snapshot

In [2]:
import os
import numpy as np
import pandas as pd

df = pd.read_csv("../data/raw/itsm_raw.csv", dtype=str, keep_default_na=False)
print(df.shape)

(46606, 25)


## Grouping the columns

In [3]:
%%writefile ../src/columns.py
id_cols       = ["Incident_ID", "KB_number", "Related_Interaction", "Related_Change"]
cat_cols      = ["CI_Name", "CI_Cat", "CI_Subcat", "WBS", "Category", "Closure_Code", "Status", "Alert_Status"]
ordinal_cols  = ["Impact", "Urgency", "Priority"]
count_cols    = ["No_of_Reassignments", "No_of_Related_Interactions", "No_of_Related_Incidents", "No_of_Related_Changes"]
time_cols     = ["Open_Time", "Reopen_Time", "Resolved_Time", "Close_Time"]
duration_cols = ["Handle_Time_hrs"]
drop_cols     = ["number_cnt"]



Overwriting ../src/columns.py


## Imports the groups and runs the 25-column check

In [4]:
import sys, importlib
importlib.invalidate_caches()
sys.path.append("../src")
from columns import *

grouped = id_cols + cat_cols + ordinal_cols + count_cols + time_cols + duration_cols + drop_cols
print(len(grouped), set(df.columns) - set(grouped))

25 set()


## placeholders-to-nulls

In [5]:
placeholders = ["", "?", "NA", "N/A", "NaN", "nan", "null", "NULL", "None", "--", "-", "NS"]
for c in df.columns:
    df[c] = df[c].str.strip().replace(placeholders, np.nan)

df.isna().sum()[lambda s: s > 0].sort_values(ascending=False)

No_of_Related_Changes         46046
Related_Change                46046
No_of_Related_Incidents       45384
Reopen_Time                   44322
Resolved_Time                  1780
Impact                         1380
Priority                       1380
Closure_Code                    460
No_of_Related_Interactions      114
CI_Cat                          111
CI_Subcat                       111
No_of_Reassignments               1
Handle_Time_hrs                   1
dtype: int64

## flags, created before any type conversion

In [6]:
df["Has_Reopen_Time"] = df["Reopen_Time"].notna().astype(int)
df["Has_Related_Change"] = df["Related_Change"].notna().astype(int)
print(df[["Has_Reopen_Time", "Has_Related_Change"]].sum())

Has_Reopen_Time       2284
Has_Related_Change     560
dtype: int64


## fix types per group, with a self-check

In [7]:
num_cols = ordinal_cols + count_cols + duration_cols
before = df[num_cols].isna().sum()

df["Urgency"] = df["Urgency"].replace({"5 - Very Low": "5"})

for c in ordinal_cols + count_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["Handle_Time_hrs"] = pd.to_numeric(df["Handle_Time_hrs"].str.replace(",", ".", regex=False), errors="coerce")

for c in time_cols:
    df[c] = pd.to_datetime(df[c], format="%d-%m-%Y %H:%M", errors="coerce")

print("New nulls created by conversion (empty means none):")
print((df[num_cols].isna().sum() - before)[lambda s: s != 0])
print(df[time_cols].isna().sum())
df.dtypes

New nulls created by conversion (empty means none):
Handle_Time_hrs    31252
dtype: int64
Open_Time            0
Reopen_Time      44322
Resolved_Time     1780
Close_Time           0
dtype: int64


CI_Name                                  str
CI_Cat                                   str
CI_Subcat                                str
WBS                                      str
Incident_ID                              str
Status                                   str
Impact                               float64
Urgency                                int64
Priority                             float64
number_cnt                               str
Category                                 str
KB_number                                str
Alert_Status                             str
No_of_Reassignments                  float64
Open_Time                     datetime64[us]
Reopen_Time                   datetime64[us]
Resolved_Time                 datetime64[us]
Close_Time                    datetime64[us]
Handle_Time_hrs                      float64
Closure_Code                             str
No_of_Related_Interactions           float64
Related_Interaction                      str
No_of_Rela

What matches expectations

* Null counts: Closure_Code is 460, as predicted. CI_Cat and CI_Subcat show 111 each, which means those are genuine blanks, not the word “Unknown”.
* Flags: Has_Reopen_Time is 2,284 and Has_Related_Change is 560.
* Datetimes: the null counts are 0, 44,322, 1,780 and 0, so the dd-mm-yyyy hh:mm format is correct for all four columns.
* Other conversions: Impact, Urgency, Priority, and the count columns converted with no new nulls.

# Shows the Handle_Time_hrs failure patterns

In [8]:
raw = pd.read_csv("../data/raw/itsm_raw.csv", dtype=str, keep_default_na=False, usecols=["Handle_Time_hrs"])
h = raw["Handle_Time_hrs"].str.strip()
conv = pd.to_numeric(h.str.replace(",", ".", regex=False), errors="coerce")
bad = h[conv.isna() & (h != "")]

print("Failing values:", len(bad))
print(bad.str.replace(r"\d", "9", regex=True).value_counts().head(10))
print("Non-digit characters:", sorted({ch for v in bad for ch in v if not ch.isdigit()}))

Failing values: 31252
Handle_Time_hrs
9,99,99,99,999    30302
9,99,999            849
99,99,999            96
9,99,99,999           5
Name: count, dtype: int64
Non-digit characters: [',']


### Handle_Time_hrs investigation

31,252 values (about two thirds) are digit-grouped integers that cannot be read as decimals. 

Decision: keep the parsed values, leave the rest null, exclude the column from all features (it is only known after handling) and use Elapsed_hrs instead. 

Evidence and tests: data_issues_log.md.

## The elapsed-time column

In [9]:
df["Elapsed_hrs"] = (df["Close_Time"] - df["Open_Time"]).dt.total_seconds() / 3600
print(df["Elapsed_hrs"].describe().round(1))
print("Close before Open:", (df["Elapsed_hrs"] < 0).sum())

count    46606.0
mean       123.1
std        452.8
min          0.0
25%          1.3
50%         18.4
75%         95.1
max      15722.6
Name: Elapsed_hrs, dtype: float64
Close before Open: 0


## null treatment per group

In [10]:
for c in ["No_of_Related_Incidents", "No_of_Related_Changes"]:
    df[c] = df[c].fillna(0)

df["Closure_Code"] = df["Closure_Code"].fillna("Not Recorded")
df["CI_Cat"] = df["CI_Cat"].fillna("Unknown")
df["CI_Subcat"] = df["CI_Subcat"].fillna("Unknown")

In [11]:
raw = pd.read_csv("../data/raw/itsm_raw.csv", usecols=["Alert_Status", "number_cnt"])
print(raw["Alert_Status"].value_counts())
print(raw["number_cnt"].astype(float).describe().round(3))
print("Unique number_cnt values:", raw["number_cnt"].nunique(), "of", len(raw))

Alert_Status
closed    46606
Name: count, dtype: int64
count    46606.000
mean         0.500
std          0.289
min          0.000
25%          0.248
50%          0.500
75%          0.749
max          1.000
Name: number_cnt, dtype: float64
Unique number_cnt values: 46606 of 46606


## Drop and save the checkpoint

Dropped: Alert_Status (constant "closed" in every row) and number_cnt
(uniform random values, all unique; not in the project brief).
Evidence is in the cell above.

In [12]:
df = df.drop(columns=["number_cnt", "Alert_Status"])
os.makedirs("../data/processed", exist_ok=True)
df.to_pickle("../data/processed/itsm_clean.pkl")
print(df.shape)
print(df.isna().sum()[lambda s: s > 0].sort_values(ascending=False))

(46606, 26)
Related_Change                46046
Reopen_Time                   44322
Handle_Time_hrs               31253
Resolved_Time                  1780
Impact                         1380
Priority                       1380
No_of_Related_Interactions      114
No_of_Reassignments               1
dtype: int64


- Handle_Time_hrs (a post-event duration in hours) has two formats: decimal commas that parse (15,353 rows with a Close_Time) and about 31,250 digit-grouped integers (pattern 9,99,99,99,999) that cannot be read as decimals.
- Relation to Open_Time and Close_Time (clean rows, by elapsed time): handle equals elapsed for 96.0% of tickets under 1 hour (10,117) but only 27.5% for 1-8 hours (1,915), 0.5% for 8-24 hours (950), 0.4% for 1-3 days (846), 0.1% for 3-7 days (708) and 0.1% over 7 days (817). Median handle time for tickets over a day is 0.5 to 5.1 hours. For 1-8 hour tickets the 75th percentile is 13.2 hours, so at least a quarter (about 480) have a handle time above their whole Open-to-Close span: handle time is not a function of Open_Time and Close_Time alone.
- Working-hours clock: among same-day weekday tickets touching time outside 7:30-18:00 (226 of 12,010), handle equals the clipped window value for 76.5% and plain elapsed time for 14.6%, so a working-day clock applies within a day (the end time is not well identified). For multi-day tickets it does not match (window match 27% to 40%, 8.9% over 7 days).
- Damaged strings: 70.2% of 31,240 agree with the window value up to a power of ten (chance 2.6%), falling from 92% (1-8 hours, 9,333 rows) through 83.5%, 72.1% and 60.5% to 36.2% (over 7 days, 6,535 rows), mirroring the clean rows. The decimal place is lost, so values cannot be recovered without a rule for the number of decimals.
- Three bins have a maximum of 99.7 to 100.0 hours among clean rows (8-24h, 1-3d, 3-7d): possibly a cap or a selection effect. Unexplained.
- Decision: unchanged. Parsed values kept, the rest null; excluded from every feature set (post-event); Elapsed_hrs derived from timestamps for descriptive use. No decoding column is built.